In [1]:
import json
import glob
import os
from pathlib import Path
import time
import os
from dotenv import load_dotenv

load_dotenv()

def consolidar_jsons(fonte, cidade, PASTA_DADOS):
    
    now = time.strftime("%Y-%m")
    
    padrao_busca = str(PASTA_DADOS / f'{cidade}_{fonte}_*.json')
    
    arquivos_json = glob.glob(padrao_busca)
    
    if not arquivos_json:
        print("Nenhum arquivo encontrado com o padrão especificado.")
        return

    dados_consolidados = []
    total_arquivos = len(arquivos_json)

    print(f"Iniciando a união de {total_arquivos} arquivos...")

    for i, caminho in enumerate(arquivos_json, 1):
        nome_base = os.path.basename(caminho)
        with open(caminho, 'r', encoding='utf-8') as f:
            try:
                conteudo = json.load(f)
                # Verifica se o conteúdo é uma lista (padrão do seu scraper)
                if isinstance(conteudo, list):
                    dados_consolidados.extend(conteudo)
                else:
                    dados_consolidados.append(conteudo)
                
                print(f"[{i}/{total_arquivos}] Adicionado: {nome_base} ({len(conteudo)} itens)")
            except Exception as e:
                print(f"Erro ao ler {nome_base}: {e}")

    # 2. Salva o arquivo final consolidado
    nome_final = f'{cidade}_{fonte}_{now}.json'
    
    caminho_final = PASTA_DADOS / nome_final

    try:
        with open(caminho_final, 'w', encoding='utf-8') as f_out:
            json.dump(dados_consolidados, f_out, indent=4, ensure_ascii=False)
        
        # --- VALIDAÇÃO DE SEGURANÇA ---
        tamanho_final = os.path.getsize(caminho_final)
        
        """if tamanho_final > 0 and len(dados_consolidados) > 0:
            print(f"✅ Consolidação concluída: {len(dados_consolidados)} registros.")
            print(f"📦 Arquivo gerado: {nome_final} ({tamanho_final / 1024 / 1024:.2f} MB)")
            
            # 4. Deleta os arquivos anteriores apenas se o final estiver OK
           
            print("🗑️ Removendo arquivos temporários (fatias)...")
            for arquivo_velho in arquivos_json:
                try:
                    os.remove(arquivo_velho)
                    print(f"   Excluído: {os.path.basename(arquivo_velho)}")
                except Exception as e:
                    print(f"   Erro ao excluir {arquivo_velho}: {e}")
        
        
            
            print("✨ Limpeza concluída com sucesso!")
        else:
            print("⚠️ Erro crítico: O arquivo final parece estar vazio. Abortando exclusão.")"""

    except Exception as e:
        print(f"❌ Erro ao salvar arquivo consolidado: {e}")


In [5]:

import asyncio
import sys
from pathlib import Path
import time

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / 'balneario_camboriu'

#consolidar_jsons('olx', 'balneario_camboriu', PASTA_DADOS)


In [6]:
PASTA_DADOS

WindowsPath('c:/Users/jefer/Documents/Ciencia-de-dados/Preco-Imoveis/dados/balneario_camboriu')

In [2]:
import pandas as pd
import warnings
import logging
import asyncio
from funcoes_limpando_dados_imoveis import (limpar_valor_iptu,
                                            limpar_banheiros, 
                                            limpar_metragem, 
                                            limpar_vagas,  
                                            #limpa_endereco_apply, 
                                            limpar_valor_condominio, 
                                            converter_para_data, 
                                            classificar_tipo_imovel, 
                                            reclassificar_outros, 
                                            preencher_todas_coordenadas,
                                            main_example, 
                                            limpar_valor_venda, 
                                            limpar_quartos, 
                                            pirabeiraba_dona_francisca, 
                                            geocodificar_dataframe,
                                            limpa_endereco_apply_zap, 
                                            limpa_endereco_apply_chave_mao, 
                                            limpa_endereco_apply_olx,)
import time
from datetime import datetime
import json
from pathlib import Path

import asyncio
import sys
from pathlib import Path
import time

cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / cidade

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logger = logging.getLogger(__name__)

warnings.filterwarnings("ignore")

start_time = time.time()

async def limpando_dados_cidades(pd_data, batch, cidade_limpeza = 'joinville', estado_limpeza = 'sc', cidade_localizacao = 'Joinville', estado_localizacao = 'SC',  tipo_async=True,  pais='Brasil'): 
       
    logger.info("Iniciando o processo de limpeza de dados de imóveis...")    
    
    pd_data = pd_data.drop_duplicates(subset=['url'])

    pd_data = pd_data[pd_data['valor_imovel'].notna()]

    pd_data_sem_nulos = pd_data.dropna(thresh=10)

    logger.info(f"Removendo linhas com muitos valores faltantes. Registros restantes: {pd_data_sem_nulos.shape}")

    #endereco_dividido = pd_data_sem_nulos['endereco'].apply(limpa_endereco_apply)

    #pd_data_endereco_dividido = pd.concat([pd_data_sem_nulos, endereco_dividido], axis=1).drop('endereco', axis=1)

    #pd_data_endereco_dividido['bairro'] = pd_data_endereco_dividido['bairro'].apply(pirabeiraba_dona_francisca)

    #logger.info("Coluna 'endereco' dividida em 'rua', 'bairro', 'cidade' e 'estado'...")
    
    pd_data_estado = pd_data_sem_nulos.copy()

    pd_data_estado = pd_data_estado[pd_data_estado['estado'] == estado_limpeza]

    logger.info(f"Removendo linhas com estado diferente de {estado_limpeza}. Registros restantes: {pd_data_estado.shape}")

    pd_data_metragem = pd_data_estado.copy()

    pd_data_metragem['metragem'] = pd_data_metragem['metragem'].apply(limpar_metragem)

    logger.info(f"Coluna 'metragem' limpa. Registros restantes: {pd_data_metragem.shape}")

    pd_data_valor_imovel = pd_data_metragem.copy()

    try:
        pd_data_valor_imovel['valor_venda'] = pd_data_valor_imovel['valor_venda'].apply(limpar_valor_venda)
    except:
        pd_data_valor_imovel['valor_imovel'] = pd_data_valor_imovel['valor_imovel'].apply(limpar_valor_venda)


    logger.info(f"Coluna 'valor_venda' limpa. Registros restantes: {pd_data_valor_imovel.shape}")

    pd_data_valor_condominio = pd_data_valor_imovel.copy()

    pd_data_valor_condominio['condominio'] = pd_data_valor_condominio['condominio'].apply(limpar_valor_condominio)

    logger.info(f"Coluna 'condominio' limpa. Registros restantes: {pd_data_valor_condominio.shape}")

    pd_data_valor_iptu = pd_data_valor_condominio.copy()

    pd_data_valor_iptu['iptu'] = pd_data_valor_iptu['iptu'].apply(limpar_valor_iptu)
    

    logger.info(f"Coluna 'iptu' limpa. Registros restantes: {pd_data_valor_iptu.shape}")

    pd_data_ano_publicacao = pd_data_valor_iptu.copy()

    pd_data_ano_publicacao['data_criacao'] = pd_data_ano_publicacao['data_criacao'].apply(converter_para_data)

    pd_data_ano_publicacao['dias_publicacao'] = (pd.to_datetime(datetime.now().strftime('%Y-%m-%d')) - pd.to_datetime(pd_data_ano_publicacao['data_criacao'], format='%d/%m/%Y')).dt.days
    
    logger.info(f"Coluna 'data_criacao' limpa. Registros restantes: {pd_data_ano_publicacao.shape}")

    pd_data_banheiros = pd_data_ano_publicacao.copy()
    
    pd_data_banheiros['banheiros'] = pd_data_banheiros['banheiros'].apply(limpar_banheiros)

    logger.info(f"Coluna 'banheiros' limpa. Registros restantes: {pd_data_banheiros.shape}")

    pd_data_quartos = pd_data_banheiros.copy()

    pd_data_quartos['quartos'] = pd_data_quartos['quartos'].apply(limpar_quartos)

    logger.info(f"Coluna 'quartos' limpa. Registros restantes: {pd_data_quartos.shape}")

    pd_data_garagem = pd_data_quartos.copy()

    #pd_data_garagem['vagas'] = pd_data_garagem['vagas'].replace('--', 0).astype('int64')

    pd_data_garagem['vagas'] = pd_data_garagem['vagas'].apply(limpar_vagas)

    logger.info(f"Coluna 'vagas' limpa. Registros restantes: {pd_data_garagem.shape}")

    pd_data_tipo_imovel = pd_data_garagem.copy()

    pd_data_tipo_imovel['tipo_imovel'] = pd_data_tipo_imovel['titulo'].apply(classificar_tipo_imovel)

    mask = pd_data_tipo_imovel['tipo_imovel'] == 'outros'

    pd_data_tipo_imovel.loc[mask, 'tipo_imovel'] = (
        pd_data_tipo_imovel.loc[mask, 'descricao']
        .apply(reclassificar_outros)
    )

    logger.info(f"Coluna 'tipo_imovel' classificada. Registros restantes: {pd_data_tipo_imovel.shape}")

    pd_data_long_lat = pd_data_tipo_imovel.copy()
    
    if tipo_async:
        pd_data_lat_log_completo = await preencher_todas_coordenadas(pd_data_long_lat, batch_size=batch, cidade=cidade_localizacao, estado=estado_localizacao, pais=pais)
    else:
        pd_data_lat_log_completo = geocodificar_dataframe(pd_data_long_lat, cidade=cidade_localizacao, estado=estado_localizacao, pais=pais)

    logger.info(f"Todas as coordenadas preenchidas. Registros restantes: {pd_data_lat_log_completo.shape}")

    pd_data_lat_log_completo['preco_por_m2'] = pd_data_lat_log_completo['valor_imovel'] / pd_data_lat_log_completo['metragem']

    logger.info(f'Coluna "preco_por_m2" criada. Registros restantes: {pd_data_lat_log_completo.shape}')

    def classificar_dentro_bairro(grupo):
        p25 = grupo["preco_por_m2"].quantile(0.25)
        p50 = grupo["preco_por_m2"].quantile(0.50)
        p75 = grupo["preco_por_m2"].quantile(0.75)
        
        def faixa(val):
            if val <= p25:
                return "barato"
            elif val <= p50:
                return "medio_baixo"
            if val <= p75:
                return "medio_alto"
            else:
                return "alto_padrao"

        grupo = grupo.copy()
        grupo["faixa"]       = grupo["preco_por_m2"].apply(faixa)
        grupo["p25_bairro"]  = p25
        grupo["p50_bairro"]  = p50
        grupo["p75_bairro"]  = p75
        return grupo

    pd_data_range_bairro_tipo_imovel = pd_data_lat_log_completo.groupby(["bairro", "tipo_imovel"], group_keys=False, ).apply(classificar_dentro_bairro)

    pd_data_range_bairro_tipo_imovel = pd.concat([pd_data_lat_log_completo, pd_data_range_bairro_tipo_imovel[['faixa', 'p25_bairro', 'p50_bairro', 'p75_bairro']]], axis=1)

    logger.info(f"Criando Faixas de preço por bairro e tipo de imóvel classificadas. Registros restantes: {pd_data_range_bairro_tipo_imovel.shape}")

    pd_data_range_bairro_tipo_imovel["desvio_mediana"] = round((pd_data_range_bairro_tipo_imovel["preco_por_m2"] - pd_data_range_bairro_tipo_imovel["p50_bairro"]) / pd_data_range_bairro_tipo_imovel["p50_bairro"],2)

    logger.info(f"Coluna 'desvio_mediana' criada. Registros restantes: {pd_data_range_bairro_tipo_imovel.shape}")
    
    return pd_data_range_bairro_tipo_imovel

In [3]:
def carregar_json(pasta_dados: Path, glob_pattern: str) -> tuple[pd.DataFrame, Path | None]:
    """
    Busca o arquivo mais recente pelo padrão e retorna um DataFrame.
    Retorna DataFrame vazio se não encontrar nenhum arquivo.
    """
    arquivos = list(pasta_dados.glob(glob_pattern))

    if not arquivos:
        logger.warning(f"Nenhum arquivo encontrado para o padrão: {glob_pattern}")
        return pd.DataFrame(), None

    arquivo = max(arquivos, key=lambda f: f.stem.split('_')[-1])
    
    logger.info(f"Arquivo encontrado: {arquivo.name}")

    try:
        with open(arquivo, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return pd.DataFrame(data), arquivo

    except Exception as e:
        logger.error(f"Erro ao carregar {arquivo.name}: {e}")
        return pd.DataFrame(), arquivo

def deletar_arquivo(arquivo: Path | None):
    if arquivo and arquivo.exists():
        arquivo.unlink()
        logger.info(f"Arquivo deletado: {arquivo.name}")
        
def carregar_parquet(pasta_dados: Path, glob_pattern: str) -> tuple[pd.DataFrame, Path | None]:
    """
    Busca o arquivo .parquet mais recente pelo padrão e retorna um DataFrame.
    """
    # Garante que estamos buscando arquivos .parquet se o pattern não especificar
    if not glob_pattern.endswith('.parquet'):
        glob_pattern = glob_pattern.replace('.json', '.parquet')

    arquivos = list(pasta_dados.glob(glob_pattern))

    if not arquivos:
        logger.warning(f"Nenhum arquivo encontrado para o padrão: {glob_pattern}")
        return pd.DataFrame(), None

    # Ordena para pegar o mais recente (mantendo sua lógica de data no final do nome)
    try:
        arquivo = max(arquivos, key=lambda f: f.stem.split('_')[-1])
    except Exception:
        arquivo = max(arquivos, key=lambda f: f.stat().st_mtime) # Fallback para data de modificação
    
    logger.info(f"Arquivo Parquet encontrado: {arquivo.name}")

    try:
        # No Parquet, o pandas lê o arquivo diretamente pelo caminho
        df = pd.read_parquet(arquivo)
        return df, arquivo

    except Exception as e:
        logger.error(f"Erro ao carregar {arquivo.name}: {e}")
        return pd.DataFrame(), arquivo

def deletar_arquivo(arquivo: Path | None):
    # Esta função permanece igual, pois Path.unlink() deleta qualquer tipo de arquivo
    if arquivo and arquivo.exists():
        arquivo.unlink()
        logger.info(f"Arquivo deletado: {arquivo.name}")

In [4]:
def normalizar_bairros(bairro, mapeamento):
    if not isinstance(bairro, str):
        return bairro
        
    bairro_low = bairro.lower()
    
    for nome_correto, variacoes in mapeamento.items():
        # Verifica se qualquer uma das variações está contida no nome original
        if any(v in bairro_low for v in variacoes):
            return nome_correto
            
    return bairro 

async def limpando_dados(name_arquivo_zap : str, 
         name_arquivo_vivareal: str, 
         name_arquivo_chave_mao: str,
         name_arquivo_olx: str,
         name_arquivo_saida: str,
         pasta_dados : Path, 
         batch: int = 1,
         tipo_async: bool = False,
         cidade_localizacao: str = 'Joinville', 
         cidade_limpeza: str = 'joinville',
         estado_limpeza: str = 'sc', 
         estado_localizacao: str = 'SC',
         pais: str = 'Brasil', 
         MAPA_BAIRROS: dict = None,):
    
    logger.info(f"Iniciando limpeza de dados de imóveis de {cidade_limpeza}...")
    
    logger.info(f"Pasta de dados: {pasta_dados}")

    pasta_dados.mkdir(parents=True, exist_ok=True)

    df_zap, arquivo_zap      = carregar_json(pasta_dados, name_arquivo_zap)
    
    df_vivareal, arquivo_vivareal = carregar_json(pasta_dados,name_arquivo_vivareal)
    
    df_chave_mao, arquivo_chave_mao = carregar_json(pasta_dados,name_arquivo_chave_mao)
    
    df_olx, arquivo_olx = carregar_json(pasta_dados,name_arquivo_olx)

    if not df_zap.empty:
        df_zap['fonte'] = 'zap_imoveis'
    if not df_vivareal.empty:
        df_vivareal['fonte'] = 'viva_real'

    if not df_chave_mao.empty:
        df_chave_mao['fonte'] = 'chave_mao'
    
    if not df_olx.empty:
        df_olx['fonte'] = 'olx'

    if df_zap.empty and df_vivareal.empty and df_chave_mao.empty and df_olx.empty:
        logger.error("Nenhum dado encontrado em nenhuma das fontes — abortando.")
        return
    
    #df_zap_endereco_limpo = df_zap['endereco'].apply(limpa_endereco_apply_zap)
    #df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(limpa_endereco_apply_zap)
    #df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(limpa_endereco_apply_chave_mao)
    #df_olx_endereco_limpo = df_olx['endereco'].apply(limpa_endereco_apply_olx)
    
    df_zap_endereco_limpo = df_zap['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
    df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
    df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(lambda x: limpa_endereco_apply_chave_mao(x, cidade_limpeza, estado_limpeza))
    df_olx_endereco_limpo = df_olx['endereco'].apply(lambda x: limpa_endereco_apply_olx(x, cidade_limpeza, estado_limpeza))
    
    df_zap_endereco =  pd.concat([df_zap, df_zap_endereco_limpo], axis=1)
    df_vivareal_endereco =  pd.concat([df_vivareal, df_vivareal_endereco_limpo], axis=1)
    df_chave_mao_endereco =  pd.concat([df_chave_mao, df_chave_mao_endereco_limpo], axis=1)
    df_olx_endereco =  pd.concat([df_olx, df_olx_endereco_limpo], axis=1)

    df = pd.concat([df_zap_endereco if not df_zap_endereco.empty else pd.DataFrame(), 
                    df_vivareal_endereco if not df_vivareal_endereco.empty else pd.DataFrame(),
                    df_chave_mao_endereco if not df_chave_mao_endereco.empty else pd.DataFrame(),
                    df_olx_endereco if not df_olx_endereco.empty else pd.DataFrame()], 
                   axis=0, ignore_index=True)
    
    logger.info(f"Total de registros carregados: {len(df)} (zap: {len(df_zap)} | vivareal: {len(df_vivareal)} | chave_mao: {len(df_chave_mao)} | olx: {len(df_olx)})")

    # Limpeza
    df_limpo = await limpando_dados_cidades(df, 
                                        batch = batch, 
                                        cidade_limpeza= cidade_limpeza, 
                                        cidade_localizacao= cidade_localizacao,
                                        tipo_async=tipo_async,
                                        estado_limpeza= estado_limpeza,
                                        estado_localizacao= estado_localizacao, 
                                        pais= pais
                                        )

    # Remove duplicatas
    colunas_dedup = ['valor_imovel', 'rua', 'bairro', 'metragem', 'quartos', 'preco_por_m2', 'banheiros', 'lat', 'lng']
    
    colunas_dedup = [c for c in colunas_dedup if c in df_limpo.columns]  

    antes = len(df_limpo)
    
    df_limpo = df_limpo.drop_duplicates(subset=colunas_dedup, keep='first').reset_index(drop=True)
    
    logger.info(f"Duplicatas removidas: {antes - len(df_limpo)} | Registros finais: {len(df_limpo)}")
    
    if MAPA_BAIRROS:
        df_limpo['bairro'] = df_limpo['bairro'].apply(normalizar_bairros, args=(MAPA_BAIRROS,))
    
    logger.info(f"Coluna 'bairro' corrigida...")
    
    #df_limpo = df_limpo.groupby('bairro').filter(lambda x: len(x) > 1)
    
    return df_limpo




In [11]:
import sys
from pathlib import Path
import time

cidade = 'sao_paulo/itaim-bibi'

cidade_limpeza = 'sao paulo'
cidade_localizacao = 'sao paulo'

estado_limpeza = 'sp'
estado_localizacao = 'sp'

pais = 'Brasil'

BASE_DIR = Path.cwd().parent

pasta_dados = BASE_DIR / 'dados' / cidade

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logger = logging.getLogger(__name__)

warnings.filterwarnings("ignore")

start_time = time.time()


name_arquivo_zap = 'sao_paulo_itaim-bibi_zap_2026-05.parquet'
name_arquivo_vivareal = 'sao_paulo_itaim-bibi_vivareal_2026-05.parquet'
name_arquivo_chave_mao = 'sao_paulo_itaim-bibi_chave_mao_2026-05.parquet'
name_arquivo_olx = 'sao_paulo_itaim-bibi_olx_2026-05.json'

df_zap, arquivo_zap      = carregar_parquet(pasta_dados, name_arquivo_zap)
    
df_vivareal, arquivo_vivareal = carregar_parquet(pasta_dados,name_arquivo_vivareal)
    
df_chave_mao, arquivo_chave_mao = carregar_parquet(pasta_dados,name_arquivo_chave_mao)
    
df_olx, arquivo_olx = carregar_json(pasta_dados,name_arquivo_olx)

if not df_zap.empty:
    df_zap['fonte'] = 'zap_imoveis'
if not df_vivareal.empty:
    df_vivareal['fonte'] = 'viva_real'

if not df_chave_mao.empty:
    df_chave_mao['fonte'] = 'chave_mao'
    
if not df_olx.empty:
    df_olx['fonte'] = 'olx'


    
df_zap_endereco_limpo = df_zap['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(lambda x: limpa_endereco_apply_chave_mao(x, cidade_limpeza, estado_limpeza))
df_olx_endereco_limpo = df_olx['endereco'].apply(lambda x: limpa_endereco_apply_olx(x, cidade_limpeza, estado_limpeza))
    
df_zap_endereco =  pd.concat([df_zap, df_zap_endereco_limpo], axis=1)
df_vivareal_endereco =  pd.concat([df_vivareal, df_vivareal_endereco_limpo], axis=1)
df_chave_mao_endereco =  pd.concat([df_chave_mao, df_chave_mao_endereco_limpo], axis=1)
df_olx_endereco =  pd.concat([df_olx, df_olx_endereco_limpo], axis=1)

df = pd.concat([df_zap_endereco if not df_zap_endereco.empty else pd.DataFrame(), 
                    df_vivareal_endereco if not df_vivareal_endereco.empty else pd.DataFrame(),
                    df_chave_mao_endereco if not df_chave_mao_endereco.empty else pd.DataFrame(),
                    df_olx_endereco if not df_olx_endereco.empty else pd.DataFrame()], 
                   axis=0, ignore_index=True)

2026-05-09 11:16:46,067 - INFO - Arquivo Parquet encontrado: sao_paulo_itaim-bibi_zap_2026-05.parquet
2026-05-09 11:16:46,078 - INFO - Arquivo Parquet encontrado: sao_paulo_itaim-bibi_vivareal_2026-05.parquet
2026-05-09 11:16:46,093 - INFO - Arquivo Parquet encontrado: sao_paulo_itaim-bibi_chave_mao_2026-05.parquet
2026-05-09 11:16:46,274 - INFO - Arquivo encontrado: sao_paulo_itaim-bibi_olx_2026-05.json


In [12]:
df_limpo = await limpando_dados_cidades(df, 
                                        batch = 100, 
                                        cidade_limpeza= cidade_limpeza, 
                                        cidade_localizacao= cidade_localizacao,
                                        tipo_async=True,
                                        estado_limpeza= estado_limpeza,
                                        estado_localizacao= estado_localizacao, 
                                        pais= pais
                                        )

2026-05-09 11:17:16,620 - INFO - Iniciando o processo de limpeza de dados de imóveis...
2026-05-09 11:17:16,778 - INFO - Removendo linhas com muitos valores faltantes. Registros restantes: (40457, 22)
2026-05-09 11:17:16,810 - INFO - Removendo linhas com estado diferente de sp. Registros restantes: (40418, 22)
2026-05-09 11:17:16,888 - INFO - Coluna 'metragem' limpa. Registros restantes: (40418, 22)
2026-05-09 11:17:17,008 - INFO - Coluna 'valor_venda' limpa. Registros restantes: (40418, 22)
2026-05-09 11:17:17,110 - INFO - Coluna 'condominio' limpa. Registros restantes: (40418, 22)
2026-05-09 11:17:17,178 - INFO - Coluna 'iptu' limpa. Registros restantes: (40418, 22)
2026-05-09 11:17:17,228 - WARNING - Erro de nome 'NoneType' object has no attribute 'split'
2026-05-09 11:17:17,228 - WARNING - Erro de nome 'NoneType' object has no attribute 'split'
2026-05-09 11:17:17,228 - WARNING - Erro de nome 'NoneType' object has no attribute 'split'
2026-05-09 11:17:17,228 - WARNING - Erro de nom

In [13]:
df_limpo

,url,titulo,metragem,banheiros,vagas,quartos,valor_imovel,condominio,endereco,iptu,...,dias_publicacao,tipo_imovel,lat,lng,preco_por_m2,faixa,p25_bairro,p50_bairro,p75_bairro,desvio_mediana
1,https://www.zapimoveis.com.br/imovel/venda-cas...,"Casa com 2 Quartos à venda, 89m² - UMUARAMA",89.0,1,3,2,210000.0,0.0,"UMUARAMA, Itanhaém - SP",NaN,...,1415.0,casa,-23.560931,-46.781294,2359.550562,barato,2359.550562,2359.550562,2359.550562,0.00
2,https://www.zapimoveis.com.br/imovel/venda-cas...,"Casa com 5 Quartos à venda, 245m² - Vila Tesouro",245.0,3,5,5,678400.0,0.0,"Rua Lupércio da Costa Machado - Vila Tesouro, ...",NaN,...,488.0,casa,-23.172487,-45.850195,2768.979592,barato,2768.979592,2768.979592,2768.979592,0.00
6,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 3 Quartos à venda, 121m² - Lau...",121.0,3,2,3,1150000.0,850.0,"Avenida do Guacá - Lauzane Paulista, São Paulo...",310.0,...,2480.0,apartamento,-23.477938,-46.636212,9504.132231,barato,9504.132231,9504.132231,9504.132231,0.00
8,https://www.zapimoveis.com.br/imovel/venda-cas...,"Casa com 3 Quartos à venda, 140m² - Jardim Por...",140.0,2,4,3,960000.0,NaN,"Rua Wilma Vinhas Santos, 500 - Jardim Portugal...",62.0,...,72.0,casa,-23.258082,-45.883122,6857.142857,barato,6857.142857,6857.142857,6857.142857,0.00
9,https://www.zapimoveis.com.br/imovel/venda-loj...,"Loja / Salão / Ponto Comercial à venda, 165m² ...",165.0,0,0,0,350000.0,0.0,"Centro, Suzano - SP",NaN,...,473.0,comercial,-23.468554,-46.530299,2121.212121,barato,2121.212121,2121.212121,2121.212121,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43004,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,Cobertura duplex para venda tem 990 metros qua...,990.0,0,0,5,37000000.0,14000.0,"Itaim Bibi, São Paulo, SP, 04542000",75000.0,...,NaN,apartamento,-23.587947,-46.680375,37373.737374,alto_padrao,15232.558140,19700.000000,27832.512315,0.90
43005,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,Apartamento Cobertura Duplex - Itaim Bibi - 6 ...,990.0,0,0,5,37000000.0,14000.0,"Itaim Bibi, São Paulo, SP, 04542001",7500.0,...,NaN,apartamento,-23.588689,-46.683259,37373.737374,alto_padrao,15232.558140,19700.000000,27832.512315,0.90
43006,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,Apartamento Cobertura Duplex - Itaim Bibi - 6 ...,990.0,0,0,5,37000000.0,14000.0,"Itaim Bibi, São Paulo, SP, 04542001",7500.0,...,NaN,apartamento,-23.588689,-46.683259,37373.737374,alto_padrao,15232.558140,19700.000000,27832.512315,0.90
43007,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,Apartamento Cobertura Duplex - Itaim Bibi - 6 ...,990.0,0,0,5,37000000.0,14000.0,"Itaim Bibi, São Paulo, SP, 04542001",7500.0,...,NaN,apartamento,-23.588689,-46.683259,37373.737374,alto_padrao,15232.558140,19700.000000,27832.512315,0.90
